# Entrenamiento de la tesis en Colab (v2: 3 en paralelo)

1. `tesis_colab.zip` debe estar en tu Drive, en **`MyDrive/tesis`**.
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*.
3. *Entorno de ejecución → Ejecutar todas*.

Corre los 7 entrenamientos del protocolo, **3 a la vez** en la misma GPU, cuantiza cada uno a int8 y lo evalúa. La celda de avance se refresca cada minuto. Cada run terminado queda en `MyDrive/tesis/runs/`. Si Colab se desconecta, *Ejecutar todas* de nuevo: lo terminado se salta y un run ya entrenado retoma desde la cuantización.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tesis'
!mkdir -p "{DRIVE}/runs"
!nvidia-smi -L

In [ ]:
# Copia al disco local de Colab: leer 58 000 fotos desde Drive en cada época sería lentísimo.
%cd /content
!cp "{DRIVE}/tesis_colab.zip" /content/ && unzip -q -o tesis_colab.zip -d /content/tesis && rm tesis_colab.zip
%cd /content/tesis

Código actualizado respecto del zip: fotos decodificadas en RAM tras la primera época y aumentos en la GPU (mismo protocolo, más rápido).

In [ ]:
%%writefile tesis/data.py
"""Carga de imágenes con tf.data. El enmascaramiento de la superposición vive dentro del modelo
(ver train.build_model), así la versión TFLite lo aplica igual en el dispositivo."""
import keras
import tensorflow as tf

from .config import CLASSES, IMG_SIZE, SEED

AUGMENT = keras.Sequential([
    keras.layers.RandomRotation(0.03, fill_mode="constant"),
    keras.layers.RandomZoom((-0.15, 0.0)),  # recorte aleatorio
    keras.layers.RandomTranslation(0.05, 0.05, fill_mode="constant"),
    keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    keras.layers.RandomContrast(0.25),
])
# ponytail: sin capa de perspectiva nativa en Keras; RandomZoom+Translation la aproximan.


def _load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8), label  # uint8: 4x menos RAM en caché


def _jpeg(img, label):
    img = tf.image.random_jpeg_quality(img, 40, 95)  # compresión con pérdida
    return tf.cast(img, tf.float32), label


def _float(img, label):
    return tf.cast(img, tf.float32), label


def dataset(df, batch=32, train=False):
    """df con columnas path y label (nombre de clase). Imágenes float32 en [0, 255].
    Las fotos se decodifican una vez y quedan en RAM (cache); los aumentos se sortean en cada época."""
    labels = df.label.map(CLASSES.index).values
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, labels))
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if train:  # AUGMENT no va aquí sino en train.py, envolviendo al modelo: así corre en la GPU
        ds = ds.shuffle(len(df), seed=SEED).map(_jpeg, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    else:
        ds = ds.map(_float, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    return ds.prefetch(tf.data.AUTOTUNE)


In [ ]:
%%writefile tesis/train.py
"""Entrenamiento por transferencia con protocolo común (sección 5.5).

Uso:
    python -m tesis.train --model mobilenetv3 --exp A
    python -m tesis.train --model resnet50 --exp A --no-mask   # control del atajo de la superposición
"""
import argparse
import json
from pathlib import Path

import keras
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from .config import CLASSES, IMG_SIZE, MASK_H, MASK_W, MODELS, SEED
from .data import AUGMENT, dataset
from .split import experiment


def overlay_mask():
    m = np.ones((IMG_SIZE, IMG_SIZE, 1), dtype="float32")
    m[-round(IMG_SIZE * MASK_H):, -round(IMG_SIZE * MASK_W):] = 0
    return m


def build_model(name, mask=True):
    """Entrada: RGB float32 [0, 255] de IMG_SIZE². Máscara y preprocesamiento incluidos en el grafo."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = inp * overlay_mask() if mask else inp
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if name == "resnet50":
        x = keras.applications.resnet50.preprocess_input(x)
        base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    elif name == "mobilenetv3":
        base = keras.applications.MobileNetV3Large(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    elif name == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=shape, pooling="avg")
    else:
        raise ValueError(f"modelo desconocido: {name}")
    x = keras.layers.Dropout(0.2)(base(x))
    return keras.Model(inp, keras.layers.Dense(len(CLASSES), activation="softmax")(x), name=name)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=MODELS, required=True)
    ap.add_argument("--exp", choices=["A", "B"], default="A")
    ap.add_argument("--no-mask", action="store_true")
    ap.add_argument("--manifest", default="data/manifest.csv")
    ap.add_argument("--stores", default="data/stores_split.csv")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--batch", type=int, default=32)
    args = ap.parse_args()

    keras.utils.set_random_seed(SEED)
    df = pd.read_csv(args.manifest)
    parts = experiment(df[df.label.notna()], pd.read_csv(args.stores), args.exp)
    run = Path("runs") / f"{args.exp}_{args.model}{'_nomask' if args.no_mask else ''}"
    run.mkdir(parents=True, exist_ok=True)

    y = parts["train"].label.map(CLASSES.index).values
    present = np.unique(y)
    weights = dict(zip(present.tolist(), compute_class_weight("balanced", classes=present, y=y)))

    model = build_model(args.model, mask=not args.no_mask)
    # Los aumentos envuelven al modelo solo para entrenar (corren en la GPU); se guarda `model`, sin ellos.
    trainer = keras.Sequential([keras.Input((IMG_SIZE, IMG_SIZE, 3)), AUGMENT, model])
    trainer.compile(keras.optimizers.Adam(args.lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    trainer.fit(
        dataset(parts["train"], args.batch, train=True),
        validation_data=dataset(parts["val"], args.batch),
        epochs=args.epochs,
        class_weight=weights,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.3),
            keras.callbacks.CSVLogger(run / "history.csv"),
        ],
    )
    model.save(run / "model.keras")
    for k, v in parts.items():
        v.to_csv(run / f"{k}.csv", index=False)
    json.dump(vars(args), open(run / "args.json", "w"), indent=2)
    print(f"guardado en {run}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tesis/evaluate.py
"""Evalúa un run sobre su conjunto de prueba (Keras o TFLite) y escribe métricas en JSON.

Uso:
    python -m tesis.evaluate runs/A_mobilenetv3 model_int8.tflite --contracts data/contracts.csv
contracts.csv (opcional): store, label   -> una fila por convenio vigente del local.
"""
import argparse
import json
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import tensorflow as tf

from .config import CLASSES
from .data import dataset
from .metrics import report, verification


def predict(model_path, df):
    ds = dataset(df, batch=1 if model_path.suffix == ".tflite" else 64)
    if model_path.suffix != ".tflite":
        return keras.models.load_model(model_path).predict(ds)
    try:
        it = tf.lite.Interpreter(str(model_path))
        it.allocate_tensors()
    except RuntimeError:  # XNNPACK no prepara el MobileNetV3 int8: kernels estándar, mismos resultados
        it = tf.lite.Interpreter(str(model_path), experimental_op_resolver_type=(
            tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES))
        it.allocate_tensors()
    inp, out = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
    probs = []
    for x, _ in ds:
        it.set_tensor(inp, x.numpy())
        it.invoke()
        probs.append(it.get_tensor(out)[0])
    return np.array(probs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("run")
    ap.add_argument("model", nargs="?", default="model.keras")
    ap.add_argument("--contracts")
    args = ap.parse_args()

    run = Path(args.run)
    test = pd.read_csv(run / "test.csv")
    probs = predict(run / args.model, test)
    y, stores = test.label.map(CLASSES.index).values, test.store.values

    contracts = None
    if args.contracts:
        contracts = pd.read_csv(args.contracts).groupby("store").label.agg(set).to_dict()
    m = report(y, probs, stores, contracts)
    if contracts:
        m.update(verification(probs, y, stores, contracts))
    m["size_mb"] = (run / args.model).stat().st_size / 2**20

    stem = Path(args.model).stem
    np.save(run / f"probs_{stem}.npy", probs)  # para bootstrap entre configuraciones
    json.dump(m, open(run / f"metrics_{stem}.json", "w"), indent=2, default=float)
    print(json.dumps({k: m[k] for k in ("f1_macro", "recall_macro", "reject_recall", "size_mb")}, indent=2, default=float))


if __name__ == "__main__":
    main()


In [ ]:
import os, subprocess, time
from pathlib import Path
import pandas as pd
from IPython.display import clear_output

os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'  # cada proceso toma solo la memoria de GPU que usa
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
PARALLEL = 3
RUNS = [  # (modelo, experimento, sin máscara); los más lentos primero
    ('resnet50', 'A', False), ('resnet50', 'B', False),
    ('efficientnetb0', 'A', False), ('efficientnetb0', 'B', False),
    ('mobilenetv3', 'A', False), ('mobilenetv3', 'B', False),
    ('mobilenetv3', 'A', True),  # control del atajo de la superposición
]

def name_of(model, exp, nomask):
    return f"{exp}_{model}{'_nomask' if nomask else ''}"

def command(model, exp, nomask):
    name, saved = name_of(model, exp, nomask), f'{DRIVE}/runs/' + name_of(model, exp, nomask)
    if os.path.exists(f'{saved}/model.keras'):  # entrenado antes de una desconexión: se retoma
        steps = [f'mkdir -p runs && cp -r "{saved}" runs/']
    else:
        steps = [f'python -m tesis.train --model {model} --exp {exp}' + (' --no-mask' if nomask else ''),
                 f'cp -r runs/{name} "{DRIVE}/runs/"']
    steps += [f'python -m tesis.quantize runs/{name}',
              f'python -m tesis.evaluate runs/{name} model.keras',
              f'python -m tesis.evaluate runs/{name} model_int8.tflite',
              f'cp -r runs/{name} "{DRIVE}/runs/"']
    return ' && '.join(steps)

def status(name):
    h = Path(f'runs/{name}/history.csv')
    if Path(f'runs/{name}/model_int8.tflite').exists():
        return f'{name}: entrenado; evaluando'
    if not h.exists() or h.stat().st_size < 60:
        return f'{name}: época 1 en curso'
    d = pd.read_csv(h)
    return (f'{name}: {len(d)} épocas · val_accuracy última {d.val_accuracy.iloc[-1]:.3f}, '
            f'mejor {d.val_accuracy[d.val_loss.idxmin()]:.3f} · {len(d) - 1 - d.val_loss.idxmin()} sin mejorar')

Path('logs').mkdir(exist_ok=True)
pending = [r for r in RUNS if not os.path.exists(f'{DRIVE}/runs/{name_of(*r)}/metrics_model_int8.json')]
running, finished, t0 = {}, [], time.time()
while pending or running:
    while pending and len(running) < PARALLEL:
        r = pending.pop(0)
        running[name_of(*r)] = subprocess.Popen(command(*r), shell=True, stderr=subprocess.STDOUT,
                                                stdout=open(f'logs/{name_of(*r)}.log', 'w'))
    for name, p in list(running.items()):
        if p.poll() is not None:
            finished.append(f"{name}: {'listo, guardado en Drive' if p.returncode == 0 else 'FALLÓ, ver logs/' + name + '.log'}")
            del running[name]
    clear_output(wait=True)
    print(f'{(time.time() - t0) / 3600:.1f} h · en curso {len(running)} · en cola {len(pending)} · terminados {len(finished)}')
    for name in running:
        print('  ', status(name))
    for line in finished:
        print('  ', line)
    if running:
        time.sleep(60)
for line in finished:
    if 'FALLÓ' in line:
        name = line.split(':')[0]
        print(f'--- {name} (últimas líneas) ---'); print(''.join(open(f'logs/{name}.log').readlines()[-25:]))

In [ ]:
# Resumen. Ojo: el conjunto de prueba aún no tiene capturas inválidas reales, así que su recall sale 0 y
# arrastra f1_macro y reject_recall. f1_presentes promedia solo las clases con ejemplos en prueba.
import glob, json

rows = []
for f in sorted(glob.glob(f'{DRIVE}/runs/*/metrics_*.json')):
    m = json.load(open(f))
    pc = [v for v in m['per_class'].values() if v['support'] > 0]
    f1 = [2 * v['precision'] * v['recall'] / (v['precision'] + v['recall'] or 1) for v in pc]
    rows.append({'run': f.split('/')[-2], 'formato': f.split('metrics_')[-1][:-5],
                 'f1_macro': m['f1_macro'], 'f1_presentes': sum(f1) / len(f1),
                 'recall_macro': m['recall_macro'], 'reject_recall': m['reject_recall'], 'size_mb': m['size_mb']})
pd.DataFrame(rows).round(3)